In [ ]:
!pip install -q openai                           # once per Colab session

In [ ]:
import json
import functools
from google.colab import userdata
from openai import OpenAI, RateLimitError, APIConnectionError

# =========================================================
# 1. GROQ CLIENT CONFIGURATION
# =========================================================


client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),        # who you are, who pays
    base_url="https://api.groq.com/openai/v1",   # send requests to Groq, not OpenAI
)
MODEL = "openai/gpt-oss-120b"


# =========================================================
# 2. BAKERY MENU
# =========================================================

MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80,
    "PineApple cake":200,
    "RedVelvet cake":800
}


# =========================================================
# 3. SYSTEM PROMPT
# =========================================================

SYSTEM = f"""
ROLE:
You are the friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
Menu with prices in rupees: {json.dumps(MENU)}
Orders need 24 hours' notice.
All cakes have eggless versions.

RULES:
Never promise a delivery time.
If it isn't in FACTS, say "I'll check with the owner."
Remember the customer's name and order during the chat.
Use the price calculation tool whenever the customer asks
for the total price of menu items.
If any one aksed any invoice dont generate it, say "contact to our team."


FORMAT:
Under 40 words.
Friendly.
English or Telugu or hindi matching the customer.
"""


# =========================================================
# 4. LOGGING DECORATOR
# =========================================================

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper


# =========================================================
# 5. PRICE CALCULATION FUNCTION
# =========================================================

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty


history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8


def chat(user_text: str) -> str:
  history.append({"role":"user","content": user_text})
  recent = [history[0]]+history[1:][-MAX_TURNS:]
  try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.5)
        reply = response.choices[0].message.content
  except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
  except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
  history.append({"role": "assistant", "content": reply})
  return reply

print("* WElCOME TO SWEET CRUMBS BAKERY *")


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

* WElCOME TO SWEET CRUMBS BAKERY *
Bot: Hi! We have Plum Cake – ₹500, Chocolate Cake – ₹650, Cupcake – ₹80, Pineapple Cake – ₹200, Red Velvet Cake – ₹800. All eggless options available. Let me know what you'd like!
Bot: Plum Cake – ₹500  
Chocolate Cake – ₹650  
Cupcake – ₹80  
Pineapple Cake – ₹200  
Red Velvet Cake – ₹800  

All eggless versions are available!
Bot: Sure! Please contact our team for the invoice details.
Bot: The total comes to ₹5,060 for 2 cupcakes, 3 Red Velvet cakes, and 5 Plum cakes. Let me know if you'd like to place the order!
Bot: Got it, Ahrsha! Your order: 2 cupcakes, 3 Red Velvet cakes, 5 Plum cakes – total ₹5,060. Let me know when you’d like to confirm!
Bot: Thanks, Harsha! I’ve noted your order (2 cupcakes, 3 Red Velvet cakes, 5 Plum cakes) – total ₹5,060. Let me know when you’d like to confirm!
